In [3]:
! pip install z3-solver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.7/31.7 MB 53.0 MB/s eta 0:00:00


# Safety_Checker

In [1]:
"""
using Z3 to formally PROVE whether a combination of patient + drug is allowed or blocked.
"""

from z3 import Bool, Solver, And, Not, Implies, BoolVal, sat


# Facts

# Patient facts
patient_has_gastric_cancer   = Bool("patient_has_gastric_cancer")
patient_has_nsclc            = Bool("patient_has_nsclc")
patient_has_bleeding_risk    = Bool("patient_has_bleeding_risk")
patient_has_egfr_mutation    = Bool("patient_has_egfr_mutation")

# Treatment choices (only one drug at a time in these tests)
giving_bevacizumab           = Bool("giving_bevacizumab")
giving_erlotinib             = Bool("giving_erlotinib")
giving_flot_regimen          = Bool("giving_flot_regimen")

# The outcome we want Z3 to check
recommendation_is_safe       = Bool("recommendation_is_safe")


# Safety Rules
# Implies(A, B) means: "If A is true, then B must be true."

SAFETY_RULES = [

    # Rule 1: Bevacizumab + gastric cancer = UNSAFE
    # Why: AVAGAST trial (Ohtsu et al., 2011) showed no benefit + bleeding risk.
    #      This is one of the real failures of Watson for Oncology (Ross & Swetlitz, 2017).
    Implies(
        And(patient_has_gastric_cancer, giving_bevacizumab),
        Not(recommendation_is_safe)    # → it is NOT safe
    ),

    # Rule 2: Bevacizumab + any patient with bleeding risk = UNSAFE
    # Why: Bevacizumab stops blood clotting. Very dangerous if bleeding already.
    Implies(
        And(giving_bevacizumab, patient_has_bleeding_risk),
        Not(recommendation_is_safe)
    ),

    # Rule 3: Erlotinib + NSCLC + EGFR mutation = SAFE ✓
    # Why: Strong clinical trial evidence (Rosell et al., 2012, NEJM).
    Implies(
        And(patient_has_nsclc, patient_has_egfr_mutation, giving_erlotinib),
        recommendation_is_safe
    ),

    # Rule 4: FLOT regimen + gastric cancer = SAFE ✓
    # Why: Al-Batran et al. (2019), Lancet — this is the standard of care.
    Implies(
        And(patient_has_gastric_cancer, giving_flot_regimen),
        recommendation_is_safe
    ),
]


# Checker

def is_safe(scenario: dict, label: str = "") -> bool:
    """
    Check if a treatment is safe for a patient.

    Args:
        scenario: a dict of {Z3 variable: True or False}
                  describing the patient and what drug we're giving.
        label:    optional name for this test (just for printing)

    Returns:
        True if safe, False if blocked.
    """
    if label:
        print(f"\n--- {label} ---")

    solver = Solver()

    # Add the medical rules
    for rule in SAFETY_RULES:
        solver.add(rule)

    # Add the facts about this specific patient + drug
    for variable, value in scenario.items():
        solver.add(variable == BoolVal(value))

    # Ask Z3: is it possible for "recommendation_is_safe" to be True?
    solver.add(recommendation_is_safe == BoolVal(True))
    result = solver.check()

    safe = (result == sat)

    if safe:
        print("  Z3 says: SAFE — recommendation is allowed.")
    else:
        print("  Z3 says: UNSAFE — recommendation is BLOCKED.")
        # Show which rule was broken
        if scenario.get(patient_has_gastric_cancer) and scenario.get(giving_bevacizumab):
            print("  Reason: Bevacizumab is dangerous in gastric cancer.")
            print("          (AVAGAST trial failure + WfO documented error)")
        if scenario.get(giving_bevacizumab) and scenario.get(patient_has_bleeding_risk):
            print("  Reason: Patient has bleeding risk — bevacizumab is contraindicated.")
        print("  ACTION: Escalate to a senior oncologist before proceeding.")

    return safe


# Formal Proof

def prove_bevacizumab_never_safe_in_gastric_cancer():
    """
    This proves: "Is there ANY possible patient where bevacizumab is safe
    in gastric cancer?" — not just one test case, but ALL possible cases.

    If Z3 says UNSAT, it's MATHEMATICALLY IMPOSSIBLE. A proof, not a test.
    """
    print("\n" + " "*55)
    print("PROOF: Can bevacizumab EVER be safe in gastric cancer?")
    print(" "*55)

    solver = Solver()
    for rule in SAFETY_RULES:
        solver.add(rule)

    # Fix these two facts as always-true
    solver.add(patient_has_gastric_cancer == BoolVal(True))
    solver.add(giving_bevacizumab         == BoolVal(True))

    # Ask: can it ever be safe?
    solver.add(recommendation_is_safe == BoolVal(True))

    result = solver.check()

    if result == sat:
        print("Unexpected: Z3 found a scenario where it could be safe. Check rules!")
    else:
        print("Z3 result: UNSAT")
        print("PROOF: There is NO possible patient where bevacizumab is safe")
        print("in gastric cancer. This is a mathematical guarantee — not just")
        print("a test of one scenario, but ALL scenarios simultaneously.")

    print(" "*55)


if __name__ == "__main__":

    # Test 1: Safe — erlotinib for a lung cancer patient with EGFR mutation
    is_safe(
        label="Erlotinib for NSCLC + EGFR mutation (should be SAFE)",
        scenario={
            patient_has_nsclc:          True,
            patient_has_egfr_mutation:  True,
            giving_erlotinib:           True,
            patient_has_gastric_cancer: False,
            patient_has_bleeding_risk:  False,
            giving_bevacizumab:         False,
            giving_flot_regimen:        False,
        }
    )

    # Test 2: Blocked — bevacizumab for gastric cancer (the real WfO failure)
    is_safe(
        label="Bevacizumab for gastric cancer (should be BLOCKED)",
        scenario={
            patient_has_gastric_cancer: True,
            giving_bevacizumab:         True,
            patient_has_nsclc:          False,
            patient_has_egfr_mutation:  False,
            patient_has_bleeding_risk:  False,
            giving_erlotinib:           False,
            giving_flot_regimen:        False,
        }
    )

    # Test 3: Blocked — bevacizumab + patient has bleeding risk
    is_safe(
        label="Bevacizumab + patient has bleeding risk (should be BLOCKED)",
        scenario={
            patient_has_nsclc:          True,
            giving_bevacizumab:         True,
            patient_has_bleeding_risk:  True,
            patient_has_gastric_cancer: False,
            patient_has_egfr_mutation:  False,
            giving_erlotinib:           False,
            giving_flot_regimen:        False,
        }
    )

    # Test 4: Safe — FLOT for gastric cancer
    is_safe(
        label="FLOT regimen for gastric cancer (should be SAFE)",
        scenario={
            patient_has_gastric_cancer: True,
            giving_flot_regimen:        True,
            giving_bevacizumab:         False,
            patient_has_nsclc:          False,
            patient_has_egfr_mutation:  False,
            patient_has_bleeding_risk:  False,
            giving_erlotinib:           False,
        }
    )


    prove_bevacizumab_never_safe_in_gastric_cancer()



--- Erlotinib for NSCLC + EGFR mutation (should be SAFE) ---
  Z3 says: SAFE — recommendation is allowed.

--- Bevacizumab for gastric cancer (should be BLOCKED) ---
  Z3 says: UNSAFE — recommendation is BLOCKED.
  Reason: Bevacizumab is dangerous in gastric cancer.
          (AVAGAST trial failure + WfO documented error)
  ACTION: Escalate to a senior oncologist before proceeding.

--- Bevacizumab + patient has bleeding risk (should be BLOCKED) ---
  Z3 says: UNSAFE — recommendation is BLOCKED.
  Reason: Patient has bleeding risk — bevacizumab is contraindicated.
  ACTION: Escalate to a senior oncologist before proceeding.

--- FLOT regimen for gastric cancer (should be SAFE) ---
  Z3 says: SAFE — recommendation is allowed.

                                                       
PROOF: Can bevacizumab EVER be safe in gastric cancer?
                                                       
Z3 result: UNSAT
PROOF: There is NO possible patient where bevacizumab is safe
in gastric cancer

# Consent_Tracker

In [2]:
"""
Check Patient Consent an dtransparency
"""

from z3 import Bool, Solver, And, Implies, BoolVal, sat


#Declare facts

ai_was_involved            = Bool("ai_was_involved")
patient_was_told           = Bool("patient_was_told")       # Did patient know AI was used?
doctor_acknowledged        = Bool("doctor_acknowledged")    # Did doctor sign off?
consent_is_written_or_digital = Bool("consent_is_written_or_digital")  # Not just verbal
this_is_high_risk          = Bool("this_is_high_risk")      # Cancer = always high risk
record_can_be_finalised    = Bool("record_can_be_finalised")  # What we want to check


#Rules

CONSENT_RULES = [

    # Rule 1: If AI was involved - patient must be told before finalising
    # Legal basis: GDPR Article 22
    Implies(
        ai_was_involved,
        Implies(record_can_be_finalised, patient_was_told)
    ),

    # Rule 2: Doctor must ALWAYS acknowledge AI involvement before finalising
    # Legal basis: EU AI Act Article 14 (human oversight)
    Implies(
        record_can_be_finalised,
        doctor_acknowledged
    ),

    # Rule 3: High-risk decisions (cancer) need written/digital consent — not verbal
    # Legal basis: EU AI Act Article 13
    Implies(
        And(this_is_high_risk, record_can_be_finalised),
        consent_is_written_or_digital
    ),

    # Rule 4: Written consent only counts if the patient was actually told
    # (you can't sign something you weren't told about)
    Implies(
        consent_is_written_or_digital,
        patient_was_told
    ),
]


#Checker

def can_finalise(scenario: dict, label: str = "") -> bool:
    if label:
        print(f"\n--- {label} ---")

    solver = Solver()
    for rule in CONSENT_RULES:
        solver.add(rule)
    for variable, value in scenario.items():
        solver.add(variable == BoolVal(value))
    solver.add(record_can_be_finalised == BoolVal(True))

    result = solver.check()
    allowed = (result == sat)

    if allowed:
        print("  Z3 says: ALLOWED — record can be finalised.")
    else:
        print("  Z3 says: BLOCKED — cannot finalise this record.")
        s = scenario
        if s.get(ai_was_involved) and not s.get(patient_was_told):
            print("  Reason: Patient was NOT told AI was involved. (GDPR Art. 22)")
        if not s.get(doctor_acknowledged):
            print("  Reason: Doctor has NOT acknowledged AI involvement. (EU AI Act Art. 14)")
        if s.get(this_is_high_risk) and not s.get(consent_is_written_or_digital):
            print("  Reason: Cancer decisions need written/digital consent — verbal not enough.")

    return allowed


# Formal Proof

def prove_no_consent_no_finalise():
    """
    Prove: Is there ANY way to finalise without patient consent?
    Expected: No — UNSAT — mathematically impossible.
    """
    print("\n" + " "*55)
    print("PROOF: Can we ever finalise without patient consent?")
    print(" "*55)

    solver = Solver()
    for rule in CONSENT_RULES:
        solver.add(rule)
    solver.add(ai_was_involved         == BoolVal(True))
    solver.add(record_can_be_finalised == BoolVal(True))
    solver.add(patient_was_told        == BoolVal(False))  # Patient NOT told

    result = solver.check()
    if result == sat:
        print("Z3 found a loophole! Review your rules.")
    else:
        print("Z3 result: UNSAT")
        print("PROOF: It is IMPOSSIBLE to finalise without telling the patient.")
        print("This holds for every possible combination of other conditions.")
    print(" "*55)


def prove_doctor_always_needed():
    """
    Prove: Even if the patient consents, is the doctor's sign-off still required?
    Expected: Yes — UNSAT — patient consent alone is never enough.
    """
    print("\n" + " "*55)
    print("PROOF: Is doctor sign-off required even with full patient consent?")
    print(" "*55)

    solver = Solver()
    for rule in CONSENT_RULES:
        solver.add(rule)
    solver.add(ai_was_involved         == BoolVal(True))
    solver.add(patient_was_told        == BoolVal(True))   # Patient IS told
    solver.add(record_can_be_finalised == BoolVal(True))
    solver.add(doctor_acknowledged     == BoolVal(False))  # Doctor NOT acknowledged

    result = solver.check()
    if result == sat:
        print("Z3: Doctor can be skipped. Review your rules.")
    else:
        print("Z3 result: UNSAT")
        print("PROOF: Even with full patient consent, doctor sign-off is ALWAYS needed.")
        print("Patient consent alone is not enough. (EU AI Act Art. 14)")
    print(" "*55)



class ConsentRecord:
    """
    A medical record that uses Z3 to check consent before it can be saved.
    Usage:
        rec = ConsentRecord("P001", "NSCLC", "Erlotinib")
        rec.patient_gave_consent(method="digital")
        rec.doctor_signed_off("Dr. Smith")
        rec.finalise()   # Only works if Z3 says it's allowed
    """
    def __init__(self, patient_id, cancer_type, treatment):
        self.patient_id  = patient_id
        self.cancer_type = cancer_type
        self.treatment   = treatment
        self._patient_told  = False
        self._consent_method = None
        self._doctor_acked   = False
        self._finalised      = False

    def patient_gave_consent(self, method="verbal"):
        """Record that the patient was informed. method = 'verbal', 'written', or 'digital'."""
        if method not in {"verbal", "written", "digital"}:
            raise ValueError(f"'{method}' is not a valid consent method.")
        self._patient_told  = True
        self._consent_method = method
        print(f"[{self.patient_id}] Patient consent recorded via {method}.")
        if method == "verbal":
            print(f"  Warning: verbal consent may not be enough for cancer decisions.")

    def doctor_signed_off(self, doctor_id):
        """Record that the doctor acknowledged the AI recommendation."""
        self._doctor_acked = True
        print(f"[{self.patient_id}] Doctor '{doctor_id}' has acknowledged AI involvement.")

    def finalise(self):
        """Try to finalise the record. Z3 will block it if rules aren't met."""
        valid_consent = self._consent_method in {"written", "digital"}

        allowed = can_finalise(
            label=f"Finalising record for {self.patient_id}",
            scenario={
                ai_was_involved:                True,
                patient_was_told:               self._patient_told,
                doctor_acknowledged:            self._doctor_acked,
                consent_is_written_or_digital:  valid_consent,
                this_is_high_risk:              True,  # Cancer is always high-risk
            }
        )

        if not allowed:
            raise PermissionError(
                f"[{self.patient_id}] Record cannot be finalised — consent rules not met."
            )
        self._finalised = True
        print(f"[{self.patient_id}] Record finalised successfully.")


if __name__ == "__main__":

    # Test 1: Blocked — no consent, no doctor sign-off
    can_finalise(
        label="No consent, no doctor sign-off (should be BLOCKED)",
        scenario={
            ai_was_involved:               True,
            patient_was_told:              False,
            doctor_acknowledged:           False,
            consent_is_written_or_digital: False,
            this_is_high_risk:             True,
        }
    )

    # Test 2: Blocked — patient told verbally, doctor not involved
    can_finalise(
        label="Verbal consent only, no doctor ACK (should be BLOCKED)",
        scenario={
            ai_was_involved:               True,
            patient_was_told:              True,
            doctor_acknowledged:           False,
            consent_is_written_or_digital: False,   # verbal = not valid for high-risk
            this_is_high_risk:             True,
        }
    )

    # Test 3: Allowed — digital consent + doctor signed off
    can_finalise(
        label="Digital consent + doctor ACK (should be ALLOWED)",
        scenario={
            ai_was_involved:               True,
            patient_was_told:              True,
            doctor_acknowledged:           True,
            consent_is_written_or_digital: True,
            this_is_high_risk:             True,
        }
    )

    # Formal proofs
    prove_no_consent_no_finalise()
    prove_doctor_always_needed()

    # Full record demo
    print("\n\n FULL RECORD ")
    rec = ConsentRecord("P002", "NSCLC Stage III", "Durvalumab")
    rec.patient_gave_consent(method="digital")
    rec.doctor_signed_off("Dr. Patel")
    rec.finalise()



--- No consent, no doctor sign-off (should be BLOCKED) ---
  Z3 says: BLOCKED — cannot finalise this record.
  Reason: Patient was NOT told AI was involved. (GDPR Art. 22)
  Reason: Doctor has NOT acknowledged AI involvement. (EU AI Act Art. 14)
  Reason: Cancer decisions need written/digital consent — verbal not enough.

--- Verbal consent only, no doctor ACK (should be BLOCKED) ---
  Z3 says: BLOCKED — cannot finalise this record.
  Reason: Doctor has NOT acknowledged AI involvement. (EU AI Act Art. 14)
  Reason: Cancer decisions need written/digital consent — verbal not enough.

--- Digital consent + doctor ACK (should be ALLOWED) ---
  Z3 says: ALLOWED — record can be finalised.

                                                       
PROOF: Can we ever finalise without patient consent?
                                                       
Z3 result: UNSAT
PROOF: It is IMPOSSIBLE to finalise without telling the patient.
This holds for every possible combination of other conditio

#Conflict_of_Interest

In [3]:
"""
check if update is trustworthy
"""

from z3 import Bool, Solver, And, Not, Implies, BoolVal, sat


#Declare facts

author_has_money_interest  = Bool("author_has_money_interest")  # Does author have financial stake?
interest_was_declared      = Bool("interest_was_declared")       # Did they disclose it?
update_was_peer_reviewed   = Bool("update_was_peer_reviewed")    # Was it checked by someone else?
reviewer_is_independent    = Bool("reviewer_is_independent")     # Not from the same company/institution
update_is_a_removal        = Bool("update_is_a_removal")         # Removing a treatment = extra risky
update_is_allowed          = Bool("update_is_allowed")           # What we want to check


#Rules

CONFLICT_OF_INTEREST_RULES = [

    # Rule 1: If there's a financial interest - it must be peer-reviewed independently
    # Basis: ICMJE (2023) conflict of interest guidelines
    Implies(
        author_has_money_interest,
        Implies(update_is_allowed, And(update_was_peer_reviewed, reviewer_is_independent))
    ),

    # Rule 2: If there's a financial interest that was NOT declared - update is BLOCKED
    # Basis: The MSKCC/Watson for Oncology failure (Ross & Swetlitz, 2017)
    Implies(
        And(author_has_money_interest, Not(interest_was_declared)),
        Not(update_is_allowed)
    ),

    # Rule 3: Removing a treatment is extra risky - always needs peer review
    # Basis: High-stakes changes need independent validation (Topol, 2019)
    Implies(
        update_is_a_removal,
        Implies(update_is_allowed, update_was_peer_reviewed)
    ),

    # Rule 4: Peer review only counts if the reviewer is truly independent
    # (A colleague from the same conflicted institution doesn't count)
    Implies(
        update_was_peer_reviewed,
        reviewer_is_independent
    ),

    # Rule 5: If no financial interest AND not a removal - update is fine without peer review
    Implies(
        And(Not(author_has_money_interest), Not(update_is_a_removal)),
        update_is_allowed
    ),
]


#Check

def is_update_allowed(scenario: dict, label: str = "") -> bool:
    """
    Check if a knowledge base update is allowed under conflict-of-interest rules.

    Args:
        scenario: dict of {Z3 variable: True or False}
        label:    optional test name for printing

    Returns:
        True if allowed, False if rejected.
    """
    if label:
        print(f"\n--- {label} ---")

    solver = Solver()
    for rule in CONFLICT_OF_INTEREST_RULES:
        solver.add(rule)
    for variable, value in scenario.items():
        solver.add(variable == BoolVal(value))
    solver.add(update_is_allowed == BoolVal(True))

    result = solver.check()
    allowed = (result == sat)

    if allowed:
        print("  Z3 says: ALLOWED — update passes conflict-of-interest checks.")
    else:
        print("  Z3 says: REJECTED — update does not pass conflict-of-interest rules.")
        s = scenario
        if s.get(author_has_money_interest) and not s.get(interest_was_declared):
            print("  Reason: Financial interest exists but was NEVER DECLARED.")
            print("          This is exactly the Watson for Oncology failure (Ross & Swetlitz, 2017).")
        if s.get(author_has_money_interest) and not s.get(update_was_peer_reviewed):
            print("  Reason: Financial interest requires independent peer review.")
        if s.get(update_is_a_removal) and not s.get(update_was_peer_reviewed):
            print("  Reason: Removing a treatment always needs peer review (high-stakes change).")
    return allowed


#Fromal Proof

def prove_undeclared_interest_always_rejected():
    """
    Prove: If someone has an undeclared financial interest, can their update
    EVER be accepted — no matter what else is true?

    Expected: UNSAT — mathematically impossible.
    This formally captures the lesson from Watson for Oncology.
    """
    print("\n" + " "*55)
    print("PROOF: Can an undeclared financial interest ever be accepted?")
    print(" "*55)

    solver = Solver()
    for rule in CONFLICT_OF_INTEREST_RULES:
        solver.add(rule)
    solver.add(author_has_money_interest == BoolVal(True))
    solver.add(interest_was_declared     == BoolVal(False))  # NOT declared
    solver.add(update_is_allowed         == BoolVal(True))   # Can it pass?

    result = solver.check()
    if result == sat:
        print("Z3 found a way through! Review your rules.")
    else:
        print("Z3 result: UNSAT")
        print("PROOF: An undeclared financial interest ALWAYS blocks the update.")
        print("No other conditions (peer review, good intentions) can override this.")
        print("This formally encodes the lesson from Watson for Oncology.")
    print(" "*55)



if __name__ == "__main__":

    # Test 1: Clean update — no money interest, no removal
    is_update_allowed(
        label="Clean update, no conflict of interest (should be ALLOWED)",
        scenario={
            author_has_money_interest: False,
            interest_was_declared:     True,
            update_was_peer_reviewed:  False,
            reviewer_is_independent:   False,
            update_is_a_removal:       False,
        }
    )

    # Test 2: Declared interest + independent peer review
    is_update_allowed(
        label="Declared interest + independent reviewer (should be ALLOWED)",
        scenario={
            author_has_money_interest: True,
            interest_was_declared:     True,   # Declared
            update_was_peer_reviewed:  True,   # Reviewed
            reviewer_is_independent:   True,   # Independent
            update_is_a_removal:       False,
        }
    )

    # Test 3: The MSKCC/Watson scenario — interest exists but was NEVER declared
    is_update_allowed(
        label="Undeclared financial interest (the real WfO failure — should be REJECTED)",
        scenario={
            author_has_money_interest: True,
            interest_was_declared:     False,  # This is the problem
            update_was_peer_reviewed:  True,
            reviewer_is_independent:   True,
            update_is_a_removal:       False,
        }
    )

    # Test 4: Non-independent reviewer (e.g. same institution)
    is_update_allowed(
        label="Declared interest but reviewer is NOT independent (should be REJECTED)",
        scenario={
            author_has_money_interest: True,
            interest_was_declared:     True,
            update_was_peer_reviewed:  True,
            reviewer_is_independent:   False,  # Colleague, not independent
            update_is_a_removal:       False,
        }
    )

    # Formal proof
    prove_undeclared_interest_always_rejected()



--- Clean update, no conflict of interest (should be ALLOWED) ---
  Z3 says: ALLOWED — update passes conflict-of-interest checks.

--- Declared interest + independent reviewer (should be ALLOWED) ---
  Z3 says: ALLOWED — update passes conflict-of-interest checks.

--- Undeclared financial interest (the real WfO failure — should be REJECTED) ---
  Z3 says: REJECTED — update does not pass conflict-of-interest rules.
  Reason: Financial interest exists but was NEVER DECLARED.
          This is exactly the Watson for Oncology failure (Ross & Swetlitz, 2017).

--- Declared interest but reviewer is NOT independent (should be REJECTED) ---
  Z3 says: REJECTED — update does not pass conflict-of-interest rules.

                                                       
PROOF: Can an undeclared financial interest ever be accepted?
                                                       
Z3 result: UNSAT
PROOF: An undeclared financial interest ALWAYS blocks the update.
No other conditions (peer rev